# Interactive Wavelength Solution

[Chapter 9](09-line-identification.ipynb) ended with 69 detected arc lines and six of them
identified by hand against the observatory's reference figure. Six (pixel, wavelength) pairs are
not a wavelength solution, but they are the part of one that needs a person.

Identification and the solution are circular: a good solution predicts each line's position to a
fraction of a pixel and makes matching unambiguous, and a good set of matches makes the fit easy.
The way in is to bootstrap from the anchors, and in `specreduce.wavecal1d.WavelengthCalibration1D`
that takes two calls. `fit_lines` fits a polynomial through the anchors and matches every
detection to the catalog line predicted nearest to it; `refine_fit` refits through all the matched
pairs and matches again at a tighter tolerance. With anchors that are strong, isolated and spread
over the detector, one call of each is the whole solution. This chapter runs them one at a time
and looks at the result after each. [Chapter 11](11-wavelength-solution-automated.ipynb) does the
same job without the human, from prior knowledge of the solution.

In [ ]:
import sys
import warnings

sys.path.append("../src")

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.modeling import models
from specreduce.background import Background
from specreduce.extract import BoxcarExtract
from specreduce.tracing import FitTrace, FlatTrace
from specreduce.tilt_solution import TiltSolution
from specreduce.wavecal1d import WavelengthCalibration1D
from specreduce.wavesol1d import WavelengthSolution1D

from common import (
    ANCHOR_PIXELS,
    ANCHOR_WAVELENGTHS,
    GTC_HGAR,
    TILT_SOLUTION_FILE,
    WAVELENGTH_SOLUTION_FILE,
    fit_tilt_solution,
    read_data,
)

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
nx = obj.data.shape[1]

## Where Chapter 9 left off

The inputs are Chapter 9's: 1D arc spectra extracted at the reference row of the rectified frames,
and the six anchors from `common.py`. The extraction is repeated here so the chapter runs on its
own; the line detection is not, because the calibration class runs it.

In [ ]:
reference = FlatTrace(arcs[0], 135)

if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

rectified_arcs = [solution.resample(frame) for frame in arcs]
rectified_obj = solution.resample(obj)


spectra = [
    BoxcarExtract(frame, reference, width=15).spectrum for frame in rectified_arcs
]

## Setting up the calibration

`WavelengthCalibration1D` holds the observations, the catalog lines and the fitted model. The
observations go in one of two ways. `arc_spectra` takes extracted 1D spectra, and the class finds
the lines itself with `find_lines`; this is what most reductions do and what this chapter does.
`obs_lines` takes line positions already measured with a detection routine of your own, one array
per arc and optionally with amplitudes; it needs `pix_bounds` and `ref_pixel` given explicitly and
forgoes the plots that draw the spectra.

`line_lists` takes one entry per arc spectrum, either `pypeit` ion names, as in Chapter 9, or an
array of wavelengths. We start with one lamp and an array, `GTC_HGAR`, the observatory's HgAr list
for R1000R.

In [ ]:
calibration = WavelengthCalibration1D(
    arc_spectra=[spectra[0]],
    line_lists=[GTC_HGAR],
    ref_pixel=512,
    unit=u.angstrom,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # unconverged refits on non-line detections
    calibration.find_lines(fwhm=2.5, noise_factor=15)

print(f"observed lines: {[len(lines) for lines in calibration.observed_lines]}")
print(f"catalog lines: {[len(lines) for lines in calibration.catalog_lines]}")

The GTC list holds the eleven lines this setup shows, and Chapter 9 found a detection for all but
the blended Hg pair near 5780 Å. `find_lines` runs `find_arc_lines` on each spectrum with Chapter
9's `fwhm` and `noise_factor` and stores the result in `observed_lines`. It applies no width cut,
so the detections Chapter 9 rejected are still there; the matching tolerance keeps them out of the
fit.

:::{note}
`observed_lines` can also be assigned directly, in the format `obs_lines` takes, which is how
positions from another detection routine go in when the spectra are still wanted for plotting.
:::

Before any fit, the two lists side by side: the catalog in wavelength space on top, the detections
in pixel space below, with nothing yet relating them.

In [ ]:
fig = calibration.plot_fit(figsize=(8, 4.2), label_kwargs=dict(size=6))

# Widen the wavelength axis so the outermost catalog lines do not sit on the spines.
fig.axes[0].set_xlim(GTC_HGAR.min() - 150, GTC_HGAR.max() + 150);

## Fitting the anchors

`fit_lines` fits the polynomial through the six anchors. With `match_obs` and `match_cat` it
**snaps** the supplied values to the nearest observed and catalog entries, so the anchor pixels,
read off a plot to the nearest whole number, are replaced by the measured centroids. Six lines are
enough for a cubic, and we use `degree=3` throughout.

In [ ]:
calibration.fit_lines(
    pixels=ANCHOR_PIXELS,
    wavelengths=ANCHOR_WAVELENGTHS,
    degree=3,
    match_obs=True,
    match_cat=True,
    refine_fit=True,
)

print(
    f"rms: {calibration.rms('wavelength'):.3f} A  ({calibration.rms('pixel'):.3f} pix)"
)
print(
    f"coverage: {calibration.solution.pix_to_wav(0):.0f} to "
    f"{calibration.solution.pix_to_wav(nx - 1):.0f} A"
)
print(
    f"dispersion at center: "
    f"{calibration.solution.pix_to_wav(513) - calibration.solution.pix_to_wav(512):.3f} A/pix"
)

That is already a good solution. Its coverage, 5097–10257 Å, matches the blue end of the grism's
documented 5100–10000 Å to 3 Å, although nothing in the fit knew that figure. The red end runs
about 260 Å past the nominal range because the cubic is extrapolated from the reddest anchor, at
9123 Å, to the last pixel, so treat it as extrapolation. The rms is 0.48 Å, 0.089 pixels.

`fit_lines` did more than fit six points. With `refine_fit=True`, its default, it matched every
detection to the catalog line predicted within `refine_max_distance` of it, 5 Å by default, and
refitted with everything it found.

In [ ]:
calibration.plot_residuals(space="wavelength");

## Refining

`refine_fit` refits through all the matched pairs and matches again at `max_match_distance`,
repeating until the matched set stops changing. The 5 Å inside `fit_lines` was generous, to catch
lines from a six-point solution; with the rms under half an Ångström we tighten to 2 Å, a few times
the rms and well under the spacing of neighboring catalog lines.

In [ ]:
calibration.refine_fit(max_match_distance=2.0)
calibration.remove_unmatched_lines()

print(f"lines retained: {[len(lines) for lines in calibration.observed_lines]}")
print(
    f"final rms: {calibration.rms('wavelength'):.3f} A "
    f"({calibration.rms('pixel'):.3f} pix)"
)

fig = calibration.plot_fit(figsize=(8, 4.2), obs_to_wav=True, label_kwargs=dict(size=6))
for ax in fig.axes:
    ax.set_xlim(GTC_HGAR.min() - 150, GTC_HGAR.max() + 150)

Nothing changed: the same nine lines at the same 0.48 Å, the normal outcome with good anchors, where
the refinement confirms the solution rather than moving it. `remove_unmatched_lines` drops the
detections that never found a partner, and with `obs_to_wav=True` the observed lines are drawn
through the solution in wavelength space, where they land on their catalog counterparts. The two
GTC lines not used, 5769.598 and 5790.663 Å, are the blended Hg pair near pixels 157 and 161, which
no single Gaussian fits.

## Using more than one lamp

HgAr alone covers the range unevenly, with nine lines and none between pixels 87 and 405.
`arc_spectra` and `line_lists` both accept sequences, so adding the other two lamps changes the
arguments, not the workflow: the same `fit_lines` with the same six HgAr anchors, then the same
`refine_fit` at 2 Å. The solution is one polynomial shared by all three spectra, so the HgAr
anchors predict where every Ne and Xe catalog line falls. Ne and Xe have no GTC list, so we use
`pypeit` ion names, with `wave_air=True` because `pypeit` wavelengths are in vacuum and the GTC
list is in air.

In [ ]:
multi = WavelengthCalibration1D(
    arc_spectra=spectra,
    line_lists=[GTC_HGAR, ["NeI"], ["XeI"]],
    line_list_bounds=(5000, 10500),
    ref_pixel=512,
    unit=u.angstrom,
    wave_air=True,
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # unconverged refits on non-line detections
    multi.find_lines(fwhm=2.5, noise_factor=15)

multi.fit_lines(
    pixels=ANCHOR_PIXELS,
    wavelengths=ANCHOR_WAVELENGTHS,
    degree=3,
    match_obs=True,
    match_cat=True,
    refine_fit=True,
)
multi.refine_fit(max_match_distance=2.0)

# Record which detections the refinement matched, and their amplitudes, before the
# unmatched ones are dropped; the amplitude comparison below needs both groups.
matched_masks = [
    ~np.ma.getmaskarray(np.ma.asarray(lines))[:, 0] for lines in multi.observed_lines
]
amplitudes = [np.asarray(lines.data)[:, 1] for lines in multi.observed_lines]
matched_per_lamp = [int(mask.sum()) for mask in matched_masks]
total_detected = sum(len(lines) for lines in multi.observed_lines)
catalog_per_lamp = [len(lines) for lines in multi.catalog_lines]
multi.remove_unmatched_lines()

print(f"catalog lines: {dict(zip(lamps, catalog_per_lamp))}")
print(f"matched:         {dict(zip(lamps, matched_per_lamp))}")
print(f"total matched:   {sum(matched_per_lamp)} of {total_detected}")
print(f"rms: {multi.rms('wavelength'):.3f} A ({multi.rms('pixel'):.3f} pix)")
print(
    f"coverage: {multi.solution.pix_to_wav(0):.0f} to "
    f"{multi.solution.pix_to_wav(nx - 1):.0f} A"
)

Two calls again, and 25 matched lines across the three lamps. The three-lamp rms is **higher** than
the single-lamp one, 0.57 Å against 0.48, yet it is the better solution. It has to satisfy lines
across the whole detector rather than nine clustered in its red half, including fainter lines with
less precise centroids, whereas the cubic through nine points is barely constrained where HgAr has
none. **Do not choose a solution by its rms alone**: coverage and line count matter at least as
much. The two solutions agree on the endpoints to within a few Ångström.

Only 25 of 85 detections were matched. Many catalog lines are too faint to appear, some detections
have no catalog counterpart, and without a width cut the list holds the spurious detections of
Chapter 9. The amplitudes of the two groups show which is which:

In [ ]:
print(
    f"{'lamp':>5}  {'matched':>8}  {'median amp':>11}  {'unmatched':>10}  {'median amp':>11}"
)
for lamp, amplitude, matched in zip(lamps, amplitudes, matched_masks):
    print(
        f"{lamp:>5}  {matched.sum():>8}  {np.median(amplitude[matched]):>11.0f}  "
        f"{(~matched).sum():>10}  {np.median(amplitude[~matched]):>11.0f}"
    )

On HgAr and Xe the unmatched detections are an order of magnitude fainter than the matched ones:
weak features, blends and failed refits. Ne is the exception, because its unresolved combs of
bright lines yield detections with no single catalog counterpart. Leaving all of them out is
right, since a poor centroid drags the fit. The identified lines and the residuals for all three
lamps:

In [ ]:
fig = multi.plot_fit(figsize=(8, 7.5), obs_to_wav=True, label_kwargs=dict(size=6))
for ax in fig.axes:
    ax.set_xlim(5000, 10500)

multi.plot_residuals(space="wavelength");

The lamps divide the range: HgAr and Ne share the blue and middle, and Xe alone holds the far red.
The residuals stay within about 2 Å with no trend against wavelength, which is what a polynomial of
the right degree through correctly identified lines looks like.

## Saving the solution

Like the tilt solution, a wavelength solution is a calibration product shared by every frame taken
in the configuration. `WavelengthSolution1D.to_asdf` stores its polynomial as an analytic model,
`from_asdf` reads it back, and Chapter 12 chains it with the tilt solution. For code with its own
handling of a wavelength axis, the `gwcs` property exposes the exact model as a GWCS object, and
the `wcs` method fits the FITS grism dispersion function of
[Greisen et al. (2006)](https://ui.adsabs.harvard.edu/abs/2006A%26A...446..747G/abstract) to it and returns an
`astropy.wcs.WCS`, typed `AWAV-GRA` for air wavelengths or `WAVE-GRI` for vacuum. Here the FITS
approximation stays within 0.2 Å of the exact cubic, 0.04 pixels, across the detector.

In [ ]:
multi.solution.to_asdf(WAVELENGTH_SOLUTION_FILE)
restored = WavelengthSolution1D.from_asdf(WAVELENGTH_SOLUTION_FILE)

## Applying the solution

There are two ways to put the solution on a spectrum. `resample` rebins a pixel-space spectrum onto
a uniform wavelength grid, conserving the integrated flux and returning a flux **density**;
`attach_wcs` leaves the samples on the pixel grid and attaches the FITS WCS. We extract the science
spectrum as Chapter 8 did and apply both.

In [ ]:
trace = FitTrace(
    rectified_obj, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
subtracted = Background.two_sided(
    rectified_obj, trace, separation=60, width=20
).sub_image()
science = BoxcarExtract(subtracted, trace, width=11).spectrum

calibrated = multi.solution.resample(science)

print(
    f"spectral axis: {calibrated.spectral_axis[0]:.1f} to {calibrated.spectral_axis[-1]:.1f}"
)
print(f"flux unit:     {calibrated.flux.unit}")

# resample returns a flux *density*, so the conserved quantity is the integral, not the
# sum. resample records the bin edges it actually used, so this integral is exact.
edges = calibrated.spectral_axis.bin_edges.to(u.AA).value
integrated = np.nansum(calibrated.flux.value * np.diff(edges))
print(f"flux retained: {integrated / np.nansum(science.flux.value):.4f}")
print(f"uncertainty:   {type(calibrated.uncertainty).__name__}")

In [ ]:
fig, axs = plt.subplots(2, figsize=(8, 4.2), constrained_layout=True)

axs[0].plot(science.spectral_axis.value, science.flux.value, lw=0.6)
axs[0].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]", ylabel="Flux [e$^-$]")

axs[1].plot(calibrated.spectral_axis.value, calibrated.flux.value, lw=0.6, c="C1")
axs[1].set(
    xlim=(calibrated.spectral_axis.value[0], calibrated.spectral_axis.value[-1]),
    xlabel=r"Wavelength [$\AA$]",
    ylabel=f"Flux [{calibrated.flux.unit.to_string('latex_inline')}]",
);

The horizontal axis is now physical: the absorption features Chapter 1 could only point at can be
looked up, the deep bands in the red being telluric O₂ and H₂O.

**The vertical axis has changed quantity.** `resample` returns a flux density in
`electron / Angstrom`, not electrons per bin, so the flux retained above is an integral rather than
a ratio of sums, and getting back to electrons means integrating, never summing. The integral takes
its widths from `spectral_axis.bin_edges`, which records the edges actually used. The default grid
here is linear, 1000 bins of 5.17 Å, but `resample` accepts any edges, and differencing the bin
centers instead gives one width too few and a spurious loss of about one part in ten thousand. With
the right widths the flux is conserved to the four decimals printed, and the uncertainty is
propagated through the rebinning, as in Chapter 6.

### Keeping the pixel grid

Resampling correlates neighboring bins, and a spectrum bound for a fitting code that models the
instrument is better left on its native pixels. `attach_wcs` returns a copy of the pixel-space
spectrum with the fitted FITS WCS as its spectral axis and the samples untouched. That is the form
to serialize: written in the `specutils` `wcs1d-fits` format, the flux is stored as an image and the
solution as header keywords that any FITS reader understands.

In [ ]:
with_wcs = multi.solution.attach_wcs(science)

## Summary

- Identification and the solution are circular, so bootstrap from a few certain anchors:
  **`fit_lines` fits them and matches the rest, and `refine_fit` refits and matches again at a
  tighter tolerance.** With good anchors, one call of each is the solution.
- **`WavelengthCalibration1D` takes arc spectra or measured line positions**: `arc_spectra` with
  `find_lines`, as most reductions do, or `obs_lines` from a detection routine of your own.
- Prefer an **instrument-specific line list**. GTC's 11-line HgAr list holds the lines this setup
  shows, so nearly every entry has a detection to match.
- `match_obs=True` and `match_cat=True` snap approximate anchors to the measured centroids and
  catalog values, so values read off a plot are good enough. Six anchors fit a cubic, and
  `fit_lines` already matches the remaining lines within 5 Å.
- **Check the coverage against the instrument documentation.** The fit reproduced the grism's
  nominal blue end to 3 Å without knowing it; the red end is extrapolation beyond the last line.
- `refine_fit` at 2 Å confirms a good solution rather than moving it; further rounds are a
  fallback for weak anchors.
- Adding lamps changes the arguments, not the workflow. The anchors of one lamp serve all of them,
  and `pypeit` lists need `wave_air=True` next to an air-wavelength list.
- **Do not choose a solution by its rms alone.** One lamp gave 0.48 Å from 9 lines in half the
  range and three lamps 0.57 Å from 25 lines over the whole detector, and the second is the better
  solution.
- **Save the solution with `to_asdf`**; `from_asdf` reads it back, and Chapter 12 chains it with
  the tilt solution. The `gwcs` property and the `wcs` method give it to other code as a GWCS or a
  FITS WCS.
- **`resample` returns a flux density** in `electron / Angstrom`, with the uncertainty propagated.
  Recover a total by integrating against `spectral_axis.bin_edges`, never by summing.
- `attach_wcs` puts the FITS WCS on a pixel-space spectrum so it can be written without
  resampling.

Next: [Automated Wavelength Solution](11-wavelength-solution-automated.ipynb), which starts from
prior knowledge of the solution instead of from anchors.